# C02 — Causal Inference II: Heterogeneous Effects & EconML

## Beyond the average treatment effect

The methods in C01 estimate a single number — the average treatment effect. But in business, you rarely care about the average. You care about **who** to treat:
- Which customers respond most to a discount?
- Which patients benefit most from a drug?
- Which markets will grow most from an expansion?

This is **heterogeneous treatment effect (HTE) estimation** — and it is the frontier where ML meets causal inference.

## The methods

| Method | Key idea | Strength |
|---|---|---|
| Causal Forest | Adapt Random Forest to estimate CATE | Non-parametric, handles interactions |
| Double ML (DML) | Partial out confounders before estimating effect | Valid with high-dim controls |
| Meta-learners (S/T/X) | Repurpose any ML model for CATE | Flexible, modular |
| Linear DML | DML with linear final stage | Interpretable HTE |

**CATE** = Conditional Average Treatment Effect = $\tau(x) = E[Y(1) - Y(0) | X = x]$

**Reference:** [EconML docs](https://econml.azurewebsites.net/)


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from econml.dml import DML, LinearDML, CausalForestDML
from econml.metalearners import SLearner, TLearner, XLearner
from econml.cate_interpreter import SingleTreeCATEInterpreter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Synthetic CATE dataset ────────────────────────────────────────────────────
# Scenario: personalized pricing experiment.
# True CATE depends on customer features: high-income customers respond more.
n = 5000

# Covariates (observed)
age = np.random.uniform(18, 70, n)
income = np.random.lognormal(10.5, 0.6, n)
tenure = np.random.exponential(24, n).clip(0, 120)
n_products = np.random.poisson(3, n)
engagement = np.random.uniform(0, 1, n)
X = np.column_stack([age, income, tenure, n_products, engagement])
X_df = pd.DataFrame(X, columns=['age','income','tenure','n_products','engagement'])

# Propensity score (treatment probability depends on features)
log_odds = -1 + 0.01*age + 0.0001*income + 0.01*tenure
propensity = 1 / (1 + np.exp(-log_odds))
T = np.random.binomial(1, propensity)  # observational (not randomized)

# True CATE: high-income, high-engagement customers benefit more
income_z = (income - income.mean()) / income.std()
engagement_z = (engagement - engagement.mean()) / engagement.std()
TRUE_CATE = 20 + 15 * income_z + 10 * engagement_z  # individual-level true effect
TRUE_ATE = TRUE_CATE.mean()

# Outcome: revenue
Y = (
    500 + 2*age + 0.001*income + 3*tenure + 50*engagement
    + TRUE_CATE * T
    + np.random.normal(0, 50, n)
)

X_train, X_test, T_train, T_test, Y_train, Y_test, cate_train, cate_test = train_test_split(
    X, T, Y, TRUE_CATE, test_size=0.3, random_state=42
)
X_df_train = pd.DataFrame(X_train, columns=X_df.columns)
X_df_test = pd.DataFrame(X_test, columns=X_df.columns)

print(f"N={n} | ATE={TRUE_ATE:.1f} | CATE range=[{TRUE_CATE.min():.1f}, {TRUE_CATE.max():.1f}]")
print(f"Treatment rate: {T.mean():.2%}")

---
## Exercise 1 — Meta-Learners: S, T, and X

## The Math

**S-Learner (Single model):**
Fit one model $\hat{\mu}(X, T)$ and estimate CATE as:
$$\hat{\tau}(x) = \hat{\mu}(x, 1) - \hat{\mu}(x, 0)$$
Risk: regularization may shrink the treatment effect toward zero.

**T-Learner (Two models):**
Fit separate models for treated and control:
$$\hat{\tau}(x) = \hat{\mu}_1(x) - \hat{\mu}_0(x)$$
Risk: extrapolation error when treated/control regions don't overlap.

**X-Learner (Hybrid):**
1. Fit T-learner to get $\hat{\mu}_0, \hat{\mu}_1$
2. Impute counterfactuals: $\tilde{D}_i^1 = Y_i - \hat{\mu}_0(X_i)$ for treated, $\tilde{D}_i^0 = \hat{\mu}_1(X_i) - Y_i$ for control
3. Fit models $\hat{\tau}_1$ on treated imputed effects, $\hat{\tau}_0$ on control imputed effects
4. Combine using propensity score: $\hat{\tau}(x) = e(x)\hat{\tau}_0(x) + (1-e(x))\hat{\tau}_1(x)$

**Task:** Train all 3 using EconML. Compare CATE RMSE against true CATE.

In [ ]:
# YOUR CODE HERE
# 1. Train SLearner, TLearner, XLearner using EconML
# 2. Predict CATE on test set
# 3. Compute RMSE vs true CATE
# 4. Return metalearner_comparison DataFrame
metalearner_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert metalearner_comparison is not None
assert len(metalearner_comparison) == 3
rmse_col = [c for c in metalearner_comparison.columns if 'rmse' in c.lower()][0]
assert (metalearner_comparison[rmse_col] > 0).all()
# All learners should produce estimates with correct sign (positive CATE mostly)
learner_col = metalearner_comparison.columns[0]
assert set(metalearner_comparison[learner_col]).issubset({'S-Learner','T-Learner','X-Learner'})
print(f"✓ Exercise 1 passed")
print(metalearner_comparison.to_string(index=False))

---
## Exercise 2 — Double Machine Learning (DML)

## The Math

DML (Chernozhukov et al. 2018) handles high-dimensional confounders using the **Frisch-Waugh-Lovell theorem**: the coefficient on $T$ in $Y \sim T + X$ equals the coefficient in the residualized regression $(Y - E[Y|X]) \sim (T - E[T|X])$.

**Algorithm:**
1. Fit ML model $\hat{E}[Y|X]$, compute residuals: $\tilde{Y} = Y - \hat{E}[Y|X]$
2. Fit ML model $\hat{E}[T|X]$, compute residuals: $\tilde{T} = T - \hat{E}[T|X]$
3. Regress $\tilde{Y}$ on $\tilde{T}$: coefficient = ATE estimate

**Cross-fitting** (crucial): fit the nuisance models on one half of data, predict on the other. This removes the regularization bias.

**Heterogeneous effects extension:** Instead of regressing $\tilde{Y}$ on $\tilde{T}$, regress on $\tilde{T} \times \theta(X)$ where $\theta(X)$ is a flexible function of covariates.

**Task:**
1. Implement DML from scratch (manual cross-fitting, 2 folds).
2. Use EconML's `LinearDML` for comparison.
3. Interpret coefficients: which features moderate the treatment effect?

In [ ]:
def dml_ate_from_scratch(Y: np.ndarray, T: np.ndarray, X: np.ndarray,
                          outcome_model=None, treatment_model=None,
                          n_folds: int = 2) -> dict:
    """
    Double ML ATE estimator with cross-fitting.
    Returns: ate_estimate, ate_se, ate_ci, Y_residuals, T_residuals
    """
    # YOUR CODE HERE
    # 1. Split into n_folds
    # 2. For each fold:
    #    a. Fit outcome model on complement, predict on fold → Y_hat
    #    b. Fit treatment model on complement, predict on fold → T_hat
    #    c. Compute residuals: Y_tilde = Y - Y_hat, T_tilde = T - T_hat
    # 3. Concatenate all residuals
    # 4. OLS: Y_tilde ~ T_tilde (no intercept) → ATE
    # 5. SE via HC3 robust standard errors
    pass

# Scratch DML
dml_scratch = dml_ate_from_scratch(
    Y_train, T_train, X_train,
    outcome_model=RandomForestRegressor(n_estimators=100, random_state=42),
    treatment_model=RandomForestClassifier(n_estimators=100, random_state=42)
)

# EconML LinearDML
dml_econml = LinearDML(
    model_y=RandomForestRegressor(n_estimators=100, random_state=42),
    model_t=RandomForestClassifier(n_estimators=100, random_state=42),
    discrete_treatment=True,
    random_state=42
)
# YOUR CODE HERE: fit dml_econml, get ATE, confidence intervals

In [ ]:
# --- ASSERTIONS ---
assert dml_scratch is not None
for k in ['ate_estimate', 'ate_se', 'ate_ci']:
    assert k in dml_scratch, f"Missing: {k}"

# Scratch DML should be close to true ATE
ate_error = abs(dml_scratch['ate_estimate'] - TRUE_ATE)
assert ate_error < 10, f"DML ATE error too large: {ate_error:.2f}"

# True ATE in confidence interval
ci = dml_scratch['ate_ci']
assert ci[0] < TRUE_ATE < ci[1], "True ATE must be in CI"

print(f"✓ Exercise 2 passed")
print(f"True ATE: {TRUE_ATE:.2f}")
print(f"DML (scratch): {dml_scratch['ate_estimate']:.2f} ± {dml_scratch['ate_se']:.2f}")
print(f"95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]")

---
## Exercise 3 — Causal Forest

## The Math

The **Causal Forest** (Wager & Athey 2018) adapts Random Forest to estimate CATEs:

Each tree splits to maximize **heterogeneity in treatment effects** rather than prediction accuracy. The CATE estimate for $x$ is a weighted average of outcomes:
$$\hat{\tau}(x) = \frac{\sum_i \alpha_i(x)(Y_i - \hat{m}(X_i))}{\sum_i \alpha_i(x)(T_i - \hat{e}(X_i))}$$

where $\alpha_i(x)$ are forest weights (how often $i$ is in the same leaf as $x$).

**Key property:** Asymptotically normal CATE estimates with valid confidence intervals — unlike standard random forests.

**Task:**
1. Train `CausalForestDML` from EconML.
2. Get CATE predictions with confidence intervals.
3. Identify the top and bottom decile of customers by CATE.
4. Profile these groups: what features drive high vs low response?
5. Return `cate_profile`: DataFrame with feature means for high/low CATE groups.

In [ ]:
# YOUR CODE HERE
cate_profile = None
cf_model = None
cate_estimates = None  # CATE predictions on test set

In [ ]:
# --- ASSERTIONS ---
assert cf_model is not None
assert cate_estimates is not None
assert cate_profile is not None

# CATE estimates should be numeric and have the right shape
assert len(cate_estimates) == len(X_test)

# CATE should be correlated with true CATE
corr = np.corrcoef(cate_estimates.flatten(), cate_test)[0, 1]
assert corr > 0.3, f"CATE estimates should correlate with truth: {corr:.3f}"

# Profile should have high and low groups
assert 'group' in cate_profile.columns or cate_profile.index.name == 'group'
print(f"✓ Exercise 3 passed — CATE correlation with truth: {corr:.3f}")
print(cate_profile)

---
## Exercise 4 — Policy Learning: Who to Treat?

## The Math

Given CATE estimates, we want to find the optimal treatment policy $\pi(x) \in \{0, 1\}$ that maximizes expected welfare:
$$\pi^*(x) = \mathbf{1}[\hat{\tau}(x) > c]$$

where $c$ is a cost threshold (treat if benefit > cost).

**Budget constraint:** If we can only treat $k$ customers, select those with the highest CATE.

**Qini curve** (generalization of ROC for uplift): sort customers by predicted CATE descending, compute cumulative incremental outcome. Area under Qini = model's ability to prioritize high-responders.

**Task:**
1. Compute the optimal policy with treatment cost = $30 per customer.
2. Compute Qini curve for Causal Forest vs random assignment vs perfect oracle.
3. Compute AUUC (Area Under Uplift Curve).
4. Return `policy_result`: dict with n_treated_optimal, expected_welfare, auuc.

In [ ]:
TREATMENT_COST = 30

def compute_qini_curve(cate_pred: np.ndarray, T: np.ndarray,
                        Y: np.ndarray) -> tuple:
    """
    Compute Qini curve.
    Sort by predicted CATE descending.
    For each fraction of population treated, compute cumulative incremental outcome.
    Returns (fractions, qini_values)
    """
    # YOUR CODE HERE
    # For each threshold k:
    #   treated_top_k = those with cate_pred >= threshold
    #   incremental = E[Y|treated in top-k, T=1] - E[Y|treated in top-k, T=0]
    pass

def auuc_score(fractions: np.ndarray, qini_values: np.ndarray) -> float:
    """Area Under Uplift Curve (normalized)."""
    # YOUR CODE HERE
    pass

# YOUR CODE HERE
policy_result = None

In [ ]:
# --- ASSERTIONS ---
assert policy_result is not None
for k in ['n_treated_optimal', 'expected_welfare', 'auuc']:
    assert k in policy_result, f"Missing: {k}"
assert policy_result['n_treated_optimal'] >= 0
assert 0 <= policy_result['auuc'] <= 1
print(f"✓ Exercise 4 passed")
print(f"Optimal to treat: {policy_result['n_treated_optimal']} customers")
print(f"Expected welfare: {policy_result['expected_welfare']:.2f}")
print(f"AUUC: {policy_result['auuc']:.4f}")

---
## Exercise 5 — Linear DML: Interpretable Heterogeneity

**Task:** Use `LinearDML` to fit a linear model for CATE:
$$\tau(X) = \theta_0 + \theta_1 \cdot \text{income} + \theta_2 \cdot \text{engagement} + \ldots$$

1. Fit `LinearDML` with all 5 features as effect moderators.
2. Extract the CATE coefficients with confidence intervals.
3. Identify which features significantly moderate the treatment effect (p < 0.05).
4. Interpret: do the coefficients match the ground truth CATE function we defined?
5. Return `linear_hte`: DataFrame with feature, coefficient, se, p_value, significant.

In [ ]:
# YOUR CODE HERE
linear_hte = None

In [ ]:
# --- ASSERTIONS ---
assert linear_hte is not None
assert len(linear_hte) >= 5  # one row per feature + intercept
for col in ['coefficient', 'se', 'p_value', 'significant']:
    assert col in linear_hte.columns, f"Missing: {col}"

# Income and engagement should be significant (they drive true CATE)
sig_features = linear_hte[linear_hte['significant']]['feature'].tolist() if 'feature' in linear_hte.columns else []
print(f"✓ Exercise 5 passed — Significant moderators: {sig_features}")
print(linear_hte.to_string(index=False))

---
## Exercise 6 — CATE Validation: R-Score & MSE

## The Math

How do you evaluate CATE estimates when you don't observe the true CATE? Two approaches:

**R-Score (Nie & Wager 2021):** Uses the DML residuals as pseudo-outcomes:
$$\text{R-score} = 1 - \frac{\text{MSE}(\hat{\tau}(X))}{\text{MSE}(0)}$$

where MSE is computed against the "R-learner" pseudo-outcomes: $\tilde{Y}/\tilde{T}$ (only valid asymptotically).

**AUTOC** (Area Under Treatment Operating Characteristic): measures how well the model ranks individuals by their treatment benefit.

**Task:**
1. Compute the R-Score for S-Learner, T-Learner, X-Learner, and Causal Forest.
2. Compute RMSE vs true CATE (possible only in simulation).
3. Check whether R-Score ranking matches RMSE ranking.
4. Return `validation_df`: model × (r_score, rmse_vs_truth) comparison.

In [ ]:
def r_score(Y_tilde: np.ndarray, T_tilde: np.ndarray,
             cate_hat: np.ndarray) -> float:
    """
    R-Score for CATE model evaluation.
    Y_tilde, T_tilde: DML residuals
    cate_hat: predicted CATEs
    Higher = better.
    """
    # YOUR CODE HERE
    # Pseudo-outcome: rho_i = (Y_tilde_i - tau_hat * T_tilde_i)^2
    # R-score = 1 - E[rho] / E[(Y_tilde - 0 * T_tilde)^2]
    pass

# YOUR CODE HERE
validation_df = None

In [ ]:
# --- ASSERTIONS ---
assert validation_df is not None
assert len(validation_df) >= 3
for col in ['r_score', 'rmse_vs_truth']:
    assert col in validation_df.columns, f"Missing: {col}"
print(f"✓ Exercise 6 passed")
print(validation_df.to_string(index=False))

---
## Exercise 7 — CATE Interpretation: Decision Tree Approximation

**Business problem:** The Causal Forest gives accurate CATEs but is a black box. Marketing needs rules like "treat customers with income > $80k and engagement > 0.7".

**Approach:** Fit a simple decision tree to the predicted CATEs — the tree provides interpretable rules.

1. Use EconML's `SingleTreeCATEInterpreter` to fit a shallow decision tree (max_depth=3) to the Causal Forest CATEs.
2. Extract the tree rules as human-readable conditions.
3. For each leaf, report: rule description, n_customers, mean_estimated_cate, mean_true_cate.
4. Return `interpretation_df`: one row per leaf.

In [ ]:
# YOUR CODE HERE
interpretation_df = None

In [ ]:
# --- ASSERTIONS ---
assert interpretation_df is not None
assert len(interpretation_df) >= 2  # at least 2 leaves
for col in ['n_customers', 'mean_estimated_cate']:
    assert col in interpretation_df.columns, f"Missing: {col}"
assert (interpretation_df['n_customers'] > 0).all()
print(f"✓ Exercise 7 passed — {len(interpretation_df)} leaf segments")
print(interpretation_df.to_string(index=False))

---
## Exercise 8 — Sensitivity Analysis: Robustness to Unmeasured Confounding

## The Math

All observational causal methods assume **no unmeasured confounders** (conditional unconfoundedness). This is untestable. Sensitivity analysis asks: *how strong would an unmeasured confounder need to be to explain away our results?*

**Rosenbaum bounds:** For a matched analysis, if there is an unmeasured confounder that multiplies the odds of treatment by at most $\Gamma$, what is the range of p-values?

**Simple partial R² approach (Cinelli & Hazlett 2020):** How much variation would an unmeasured confounder need to explain in both $T$ and $Y$ to reduce the estimated effect to zero?

**Task:**
1. Run the DiD from C01 on this dataset (simulating the pricing experiment).
2. Implement a simple sensitivity analysis: add a synthetic confounder of increasing strength and observe how the estimate changes.
3. Find the minimum confounder strength that makes the estimate insignificant.
4. Return `sensitivity_result`: DataFrame showing confounder_strength vs estimate and p_value.

In [ ]:
def sensitivity_analysis(Y: np.ndarray, T: np.ndarray, X: np.ndarray,
                           confounder_strengths: list) -> pd.DataFrame:
    """
    Sensitivity to unmeasured confounding.
    For each strength gamma:
    - Add synthetic confounder U ~ N(0,1) with effect gamma on both T and Y
    - Re-estimate ATE
    - Record estimate and p-value
    Returns DataFrame: confounder_strength, ate_estimate, p_value, significant
    """
    # YOUR CODE HERE
    pass

strengths = [0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
sensitivity_result = sensitivity_analysis(Y_train, T_train, X_train, strengths)

In [ ]:
# --- ASSERTIONS ---
assert sensitivity_result is not None
assert len(sensitivity_result) == len(strengths)
for col in ['confounder_strength', 'ate_estimate', 'p_value']:
    assert col in sensitivity_result.columns
# At strength=0 the result should be significant
assert sensitivity_result[sensitivity_result['confounder_strength']==0]['p_value'].values[0] < 0.05
print(f"✓ Exercise 8 passed")
print(sensitivity_result.to_string(index=False))

---
## Exercise 9 — Continuous Treatment: Dose-Response

## The Math

So far treatment has been binary. With a **continuous treatment** $T \in \mathbb{R}$ (e.g., ad spend), we estimate the **dose-response function** $E[Y(t)|X=x]$ and its derivative — the **marginal treatment effect**.

EconML's `DML` with continuous treatment estimates:
$$E[Y | T=t, X=x] = \theta(x) \cdot t + g(x)$$

where $\theta(x)$ is the heterogeneous effect of a one-unit increase in $T$.

**Task:** Create a continuous treatment dataset (ad spend → sales). Estimate the marginal effect of ad spend, allow it to vary by customer segment.

In [ ]:
# Continuous treatment dataset: ad spend → sales
np.random.seed(33)
n_cont = 3000
X_cont = np.random.normal(0, 1, (n_cont, 4))
T_cont = np.exp(0.5 * X_cont[:, 0] + np.random.normal(0, 0.5, n_cont))  # log-normal spend
TRUE_MARGINAL = 0.3 + 0.2 * X_cont[:, 0]  # heterogeneous marginal ROI
Y_cont = 100 + TRUE_MARGINAL * T_cont + 5 * X_cont[:, 1] + np.random.normal(0, 10, n_cont)

# YOUR CODE HERE
continuous_hte_result = None

In [ ]:
# --- ASSERTIONS ---
assert continuous_hte_result is not None
assert 'marginal_effect_estimates' in continuous_hte_result
marginal_ests = continuous_hte_result['marginal_effect_estimates']
assert len(marginal_ests) > 0
# Estimates should be positively correlated with true marginal effects
X_test_cont = X_cont[:500]
true_marginal_test = TRUE_MARGINAL[:500]
if len(marginal_ests) == 500:
    corr = np.corrcoef(marginal_ests, true_marginal_test)[0, 1]
    assert corr > 0.2, f"Marginal effects should correlate with truth: {corr:.3f}"
print(f"✓ Exercise 9 passed")
print(f"Mean estimated marginal effect: {np.mean(marginal_ests):.4f}")
print(f"True mean: {TRUE_MARGINAL.mean():.4f}")

---
## Exercise 10 — Capstone: End-to-End Causal ML Pipeline

**Business problem:** An e-commerce company wants to optimize its coupon strategy. They run an observational study where some customers received a 10% discount coupon. They want to know:
1. What was the average lift in purchases from the coupon?
2. Which customers respond most to coupons?
3. Given a budget of 1000 coupons, who should receive them?
4. How confident are we in these estimates?

Build a complete pipeline:
1. **Estimate ATE** with DML (report value and 95% CI)
2. **Estimate CATE** with Causal Forest
3. **Validate** using R-Score
4. **Policy**: identify top-1000 customers to target
5. **Interpret**: decision tree of CATE (max_depth=2)
6. **Sensitivity**: test robustness to one hidden confounder
7. Return `final_report` dict with all results and a `recommendation` string.

In [ ]:
# Synthetic coupon dataset
np.random.seed(88)
n_coupon = 4000
X_coupon = np.column_stack([
    np.random.uniform(18, 65, n_coupon),           # age
    np.random.lognormal(10.3, 0.5, n_coupon),      # income
    np.random.exponential(12, n_coupon).clip(0, 60),# months_since_signup
    np.random.poisson(5, n_coupon)                  # past_purchases
])

propensity_coupon = 0.3 + 0.1 * (X_coupon[:, 3] > 5)  # engaged users more likely to get coupon
T_coupon = np.random.binomial(1, propensity_coupon)

TRUE_CATE_COUPON = 15 + 20 * (X_coupon[:, 3] > 5) + 10 * (X_coupon[:, 1] > np.median(X_coupon[:, 1]))
Y_coupon = (
    50 + 0.5 * X_coupon[:, 3] + 0.0001 * X_coupon[:, 1]
    + TRUE_CATE_COUPON * T_coupon
    + np.random.normal(0, 20, n_coupon)
)

feat_names_coupon = ['age', 'income', 'months_since_signup', 'past_purchases']

# YOUR CODE HERE
final_report = None

In [ ]:
# --- ASSERTIONS ---
assert final_report is not None
required = ['ate_estimate', 'ate_ci', 'r_score', 'n_targeted',
            'expected_incremental_revenue', 'recommendation']
for k in required:
    assert k in final_report, f"Missing: {k}"

# ATE should be positive and significant
ate = final_report['ate_estimate']
assert ate > 0, f"ATE should be positive (coupon increases purchases)"

# True ATE in confidence interval
true_ate_coupon = TRUE_CATE_COUPON.mean()
ci = final_report['ate_ci']
assert ci[0] < true_ate_coupon < ci[1] or abs(ate - true_ate_coupon) < 10

# Target exactly 1000 customers
assert final_report['n_targeted'] == 1000

assert isinstance(final_report['recommendation'], str) and len(final_report['recommendation']) > 30

print(f"✓ Exercise 10 passed — Full causal ML pipeline complete")
print(f"ATE: {ate:.2f} (true: {true_ate_coupon:.2f})")
print(f"95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]")
print(f"R-Score: {final_report['r_score']:.4f}")
print(f"Expected incremental revenue from 1000 coupons: ${final_report['expected_incremental_revenue']:,.0f}")
print(f"Recommendation: {final_report['recommendation']}")